# 6. Geração de imagens para termos e bigramas
Este notebook é dedicado exclusivamente à criação de imagens de nuvem de palavras para os eventos WIE e WEI.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))

from apoio.funcoes import montar_frequencias_palavras, gerar_nuvem_palavras

pasta_processados = raiz / 'dados' / '1_processados'
pasta_consumo = raiz / 'dados' / '2_consumo'
pasta_imagens = pasta_consumo / 'imagens'
pasta_imagens.mkdir(parents=True, exist_ok=True)

In [ ]:
freq_termos_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_termos_ano_evento.csv', encoding='utf-8-sig')
freq_bigramas_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_bigramas_ano_evento.csv', encoding='utf-8-sig')

if 'evento' not in freq_termos_ano_evento.columns or 'evento' not in freq_bigramas_ano_evento.columns:
    raise ValueError("Os arquivos de frequência precisam conter a coluna 'evento'.")

freq_termos_ano_evento['evento'] = freq_termos_ano_evento['evento'].fillna('NA').astype(str).str.strip().replace({'': 'NA'})
freq_bigramas_ano_evento['evento'] = freq_bigramas_ano_evento['evento'].fillna('NA').astype(str).str.strip().replace({'': 'NA'})

print(f'Termos carregados: {len(freq_termos_ano_evento)}')
print(f'Bigramas carregados: {len(freq_bigramas_ano_evento)}')
print('Eventos encontrados:', sorted(freq_termos_ano_evento['evento'].unique()))

In [ ]:
for evento in sorted(freq_termos_ano_evento['evento'].dropna().astype(str).str.strip().unique()):
    frequencias_termos = montar_frequencias_palavras(freq_termos_ano_evento, 'termo', evento)
    frequencias_bigramas = montar_frequencias_palavras(freq_bigramas_ano_evento, 'bigrama', evento)

    arquivo_termos = gerar_nuvem_palavras(
        frequencias_termos,
        pasta_imagens / f'nuvem_termos_{evento.lower()}.png'
    )
    arquivo_bigramas = gerar_nuvem_palavras(
        frequencias_bigramas,
        pasta_imagens / f'nuvem_bigramas_{evento.lower()}.png'
    )

    print(f'Gerado: {arquivo_termos}')
    print(f'Gerado: {arquivo_bigramas}')